<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day07-discussion-1.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 7 — In-class discussion problem (1 of 3)

Discuss **as a group first**, write down a guess, then run the code to
check it before presenting.

## Myoglobin vs. hemoglobin: does quaternary structure change secondary structure?

Myoglobin (PDB `1MBN`, solved by Kendrew in 1958 — the very first protein
structure ever determined) is hemoglobin's evolutionary cousin: recall
Day 4's orthologs/paralogs figure, where myoglobin and the globins are
the family used as the running example. Myoglobin stores oxygen
in muscle as a **single chain** — it has no quaternary structure at all,
unlike hemoglobin's four-chain assembly.

**As a group, before running anything:** given that myoglobin and
hemoglobin's beta chain are both members of the same "globin fold"
family, do you expect myoglobin's secondary-structure composition
(percent helix, percent strand) to look **similar** to hemoglobin's beta
chain (80.8% helix, 0% strand, from the main notebook), or **different**,
given that one works alone and the other only works as part of a
four-chain complex? Write down a one-sentence prediction, then run the
cell below.

In [1]:
import subprocess
import os
from Bio.PDB import PDBList, PDBParser

pdbl = PDBList()
mb_path = pdbl.retrieve_pdb_file("1mbn", pdir=".", file_format="pdb")

pymol_script = '''
from pymol import cmd

cmd.load("PDBPATH", "mb")
cmd.dss("mb")

ss_list = []
cmd.iterate("mb and polymer and name CA",
            "ss_list.append(ss)", space={"ss_list": ss_list})

total = len(ss_list)
helix = sum(1 for s in ss_list if s == "H")
strand = sum(1 for s in ss_list if s == "S")
print("TOTAL", total)
print("HELIX", helix)
print("STRAND", strand)
print("PCT_HELIX", round(100 * helix / total, 1))
print("PCT_STRAND", round(100 * strand / total, 1))
'''.replace("PDBPATH", mb_path)

with open("_disc1_pymol.py", "w") as f:
    f.write(pymol_script)

pymol_env = dict(os.environ)
pymol_env["PATH"] = "/usr/bin:/bin:" + pymol_env.get("PATH", "")

result = subprocess.run(["/usr/bin/pymol", "-cq", "_disc1_pymol.py"],
                         capture_output=True, text=True, env=pymol_env)
for line in result.stdout.splitlines():
    if line.startswith(("TOTAL", "HELIX", "STRAND", "PCT")):
        print(line)

os.remove("_disc1_pymol.py")

TOTAL 153
HELIX 122
STRAND 0
PCT_HELIX 79.7
PCT_STRAND 0.0


**Discussion point.** Myoglobin comes out at **79.7% helix, 0%
strand** (122 of 153 residues) — essentially identical to hemoglobin's
beta chain (80.8%, 0%). Secondary and tertiary structure are properties
of *one folded chain*; they don't care whether that chain then goes on
to assemble with others. Quaternary structure is a genuinely separate,
additional level on top.

That raises the real biological question: if the fold itself doesn't
need four chains, why does hemoglobin have them? The answer is
**cooperativity** — hemoglobin needs to bind oxygen efficiently in the
lungs (high O₂ concentration) and *release* it efficiently in tissues
(low O₂ concentration), and it does this by having the four subunits
allosterically signal each other (the T↔R quaternary shift measured in
the main notebook) so that binding one subunit's oxygen makes the
others bind more easily too. Myoglobin only ever needs to store oxygen
at one fixed affinity in muscle — it has no analogous need to shift its
affinity in response to conditions, so a single chain is enough.